# VIETNAMESE TRAFFIC SIGNS DETECTION (WITH YOLO26)

## Prepare the dataset - source: https://www.kaggle.com/datasets/maitam/vietnamese-traffic-signs/data

### Download dataset

In [1]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("maitam/vietnamese-traffic-signs")

# print("Path to dataset files:", path)

In [2]:
# !ls /root/.cache/kagglehub/datasets/maitam/vietnamese-traffic-signs/versions/4/archive

In [3]:
# !mv /root/.cache/kagglehub/datasets/maitam/vietnamese-traffic-signs/versions/4/* /content/

In [4]:
!ls -l /kaggle/input/vietnamese-traffic-signs/archive/images | wc -l

3217


### Convert dataset to correct YOLOv11 format:
- train
  - images
    - example.jpg
  - labels
    - example.txt
- valid
  - images
  - labels
- test
  - images
  - labels
- data.yaml
- classes_vie.txt

And apply data augmentation to images and labels then save them

In [5]:
import os
import random
import shutil
import cv2
import albumentations as A
import yaml
from tqdm import tqdm

# Paths
src_folder = '/kaggle/input/vietnamese-traffic-signs/archive'
dst_folder = '/kaggle/working/dataset_yolov11'

# Parameters
train_ratio = 0.8
valid_ratio = 0.1
img_size = 640

# Create necessary directories
for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(dst_folder, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(dst_folder, split, 'labels'), exist_ok=True)

# Load class names
with open(os.path.join(src_folder, 'classes_vie.txt'), 'r', encoding='utf-8') as f:
    classes = [line.strip() for line in f.readlines()]

# Get images
all_images = [os.path.join(src_folder, 'images', f) for f in os.listdir(os.path.join(src_folder, 'images')) if f.endswith('.jpg')]
random.shuffle(all_images)

# Split dataset
train_size = int(len(all_images) * train_ratio)
valid_size = int(len(all_images) * valid_ratio)

train_images = all_images[:train_size]
valid_images = all_images[train_size:train_size+valid_size]
test_images = all_images[train_size+valid_size:]

# Comprehensive augmentation pipeline
augmentations = A.Compose([
    A.Resize(640, 640),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0),
    A.RandomRotate90(p=0),
    A.Rotate(limit=15, p=0.5),
    A.Affine(shear=5, p=0.3),
    A.GaussianBlur(blur_limit=(3,5), p=0.2),
    A.RandomBrightnessContrast(p=0.5),
    A.RandomGamma(p=0.2),
    A.GaussNoise(p=0.3),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
], bbox_params=A.BboxParams(format='yolo'))


def process_and_save(images, split):
    for img_path in tqdm(images, desc=f'Processing {split}'):
        filename = os.path.basename(img_path)
        label_filename = filename.replace('.jpg', '.txt')

        # Read image and labels
        image = cv2.imread(img_path)
        h, w = image.shape[:2]

        label_path = os.path.join(src_folder, 'labels', label_filename)
        with open(label_path, 'r') as f:
            labels = [list(map(float, line.split())) for line in f.readlines()]

        bboxes = [label[1:] + [int(label[0])] for label in labels]  # bbox + class_id

        # Check bbox validity
        invalid_bbox = False
        for bbox in bboxes:
            x_center, y_center, bbox_w, bbox_h, _ = bbox
            if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 <= bbox_w <= 1 and 0 <= bbox_h <= 1):
                invalid_bbox = True
                break

        if invalid_bbox:
            print(f"Skipping {filename} due to invalid bbox values.")
            continue

        try:
            if split == 'train':
                # Save original image and labels
                original_img_resized = cv2.resize(image, (640, 640))
                cv2.imwrite(os.path.join(dst_folder, split, 'images', 'original_' + filename), original_img_resized)
                with open(os.path.join(dst_folder, split, 'labels', 'original_' + label_filename), 'w') as f:
                    for bbox in bboxes:
                        cls_id = bbox[-1]
                        bbox_coords = bbox[:4]
                        f.write(f"{cls_id} {' '.join(map(str, bbox_coords))}\n")

                # Save augmented image and labels
                augmented = augmentations(image=image, bboxes=bboxes)
                img_processed, bboxes_processed = augmented['image'], augmented['bboxes']
            else:
                img_processed = cv2.resize(image, (640, 640))
                bboxes_processed = bboxes
        except ValueError as e:
            print(f"Skipping {filename} due to augmentation error: {e}")
            continue

        # Save augmented image
        cv2.imwrite(os.path.join(dst_folder, split, 'images', filename), img_processed)

        # Save augmented labels
        with open(os.path.join(dst_folder, split, 'labels', label_filename), 'w') as f:
            for bbox in bboxes_processed:
                cls_id = bbox[-1]
                bbox_coords = bbox[:4]
                f.write(f"{cls_id} {' '.join(map(str, bbox_coords))}\n")


# Execute
process_and_save(train_images, 'train')
process_and_save(valid_images, 'valid')
process_and_save(test_images, 'test')

# dataset.yaml
dataset_yaml = {
    'path': os.path.abspath(dst_folder),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': classes
}

with open(os.path.join(dst_folder, 'dataset.yaml'), 'w', encoding='utf-8') as f:
    yaml.dump(dataset_yaml, f, allow_unicode=True)


/usr/local/lib/python3.10/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
Processing train:  64%|██████▍   | 1647/2572 [01:10<00:41, 22.43it/s]

Skipping 0248.jpg due to augmentation error: Expected x_max for bbox [ 0.96562546  0.294444    1.0187504   0.396296   49.        ] to be in the range [0.0, 1.0], got 1.0187504291534424.


Processing test: 100%|██████████| 323/323 [00:05<00:00, 55.70it/s]


In [6]:
!ls -l /kaggle/working/dataset_yolov11/train/images | wc -l
!ls -l /kaggle/working/dataset_yolov11/train/labels | wc -l


5144
5144


In [7]:
dataset_yaml = {
    'train': '/kaggle/working/dataset_yolov11/train/images',
    'val': '/kaggle/working/dataset_yolov11/valid/images',
    'test': '/kaggle/working/dataset_yolov11/test/images',
    'nc': len(classes),
    'names': classes
}

with open(os.path.join(dst_folder, 'data.yaml'), 'w', encoding='utf-8') as f:
    yaml.dump(dataset_yaml, f, allow_unicode=True, default_flow_style=None)

## Load YOLOv11 model

In [8]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.8 MB/s eta 0:00:00


### Download pre-trained weights (yolo26m)

In [9]:
import torch
torch.cuda.empty_cache()

### Train

In [10]:
from ultralytics import YOLO
import numpy as np
import optuna
from ultralytics.utils import checks
checks.check_amp = lambda *args, **kwargs: True

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [11]:
model = YOLO("yolo26m.pt")

In [12]:
def objective(trial):
    lr0 = trial.suggest_float("lr0", 1e-4, 1e-1, log=True)
    lrf = trial.suggest_float("lrf", 0.01, 1.0)
    momentum = trial.suggest_float("momentum", 0.85, 0.98)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    warmup_epochs = trial.suggest_float("warmup_epochs", 1.0, 5.0)
    warmup_momentum = trial.suggest_float("warmup_momentum", 0.6, 0.95)
    box = trial.suggest_float("box", 3.0, 10.0)
    cls = trial.suggest_float("cls", 0.2, 1.0)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    
    trial_model = YOLO("yolo26m.pt")
    results = trial_model.train(
        data= "/kaggle/working/dataset_yolov11/data.yaml",  
        epochs=10,
        imgsz=640,
        batch=16,
        workers=0, 
        cache=False,
        device=0,
        lr0=lr0,
        lrf=lrf,
        momentum=momentum,
        weight_decay=weight_decay,
        warmup_epochs=warmup_epochs,
        warmup_momentum=warmup_momentum,
        box=box,
        cls=cls,
        dropout=dropout,
        project="optuna_tuning",
        name=f"trial_{trial.number}",
        exist_ok=True,
        verbose=False,
        amp=False,
    )

    metric = results.results_dict["metrics/mAP50-95(B)"]

    return metric

In [13]:
import torch
from pathlib import Path

study = optuna.create_study(direction="maximize") 
study.optimize(objective, n_trials=8)
# After optimization
print("Best trial:")
trial = study.best_trial
print(f"Value: {trial.value}")
print("Best hyperparameters:", trial.params)

# Save best parameters to .pt file
best_params = trial.params
save_path = Path("/kaggle/working/best_hyperparameters.pt")

torch.save(best_params, save_path)
print(f"Best hyperparameters saved to {save_path}")

[I 2026-05-03 15:04:17,782] A new study created in memory with name: no-name-12de5f7d-f1f6-47e9-8523-397b0cced9b0


Ultralytics 8.4.46 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=5.751160204104982, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.4745807914357741, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_yolov11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.49063013633799957, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01017880352268949, lrf=0.974550470933724, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.9422147318844769, mosaic=1.0, multi_scale=0.0, n

/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        321        849       0.91      0.893      0.969      0.768
Speed: 0.2ms preprocess, 13.1ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/optuna_tuning/trial_0


[I 2026-05-03 16:21:29,441] Trial 0 finished with value: 0.7683677049383714 and parameters: {'lr0': 0.01017880352268949, 'lrf': 0.974550470933724, 'momentum': 0.9422147318844769, 'weight_decay': 1.7145446532513767e-05, 'warmup_epochs': 2.594725928916278, 'warmup_momentum': 0.7707842008730237, 'box': 5.751160204104982, 'cls': 0.4745807914357741, 'dropout': 0.49063013633799957}. Best is trial 0 with value: 0.7683677049383714.


...

Ultralytics 8.4.46 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=9.855528033949842, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.3110478153255427, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_yolov11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1971868139973072, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0006644984981427132, lrf=0.8946763923787161, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.8792933367536439, mosaic=1.0, multi_scale=0.0,

/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        321        849      0.905      0.897      0.967      0.772
Speed: 0.2ms preprocess, 13.3ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/optuna_tuning/trial_7


[I 2026-05-04 01:21:45,990] Trial 7 finished with value: 0.7717749972120705 and parameters: {'lr0': 0.0006644984981427132, 'lrf': 0.8946763923787161, 'momentum': 0.8792933367536439, 'weight_decay': 0.0004099403899032309, 'warmup_epochs': 2.3302609484994234, 'warmup_momentum': 0.641216849434569, 'box': 9.855528033949842, 'cls': 0.3110478153255427, 'dropout': 0.1971868139973072}. Best is trial 3 with value: 0.775925186301873.


Best trial:
Value: 0.775925186301873
Best hyperparameters: {'lr0': 0.00015227133216016623, 'lrf': 0.8624742081912739, 'momentum': 0.8782687215781079, 'weight_decay': 0.000539712482371129, 'warmup_epochs': 3.6186402296231983, 'warmup_momentum': 0.8612269930812944, 'box': 7.744679389461637, 'cls': 0.5876386316157864, 'dropout': 0.3184038958936688}
Best hyperparameters saved to /kaggle/working/best_hyperparameters.pt
